# 레슨 08 — 수집 데이터 검증과 저장

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/08/%EB%A0%88%EC%8A%A8%2008%20%E2%80%94%20%EC%88%98%EC%A7%91%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EA%B2%80%EC%A6%9D%EA%B3%BC%20%EC%A0%80%EC%9E%A5.ipynb)

> 선생님용 강의 노트북이다. 코랩에서 확인하려면 우측 상단 또는 아래의 **Open in Colab** 버튼을 클릭한다.


> 코랩에서 실행하려면 우측 상단 또는 아래의 **Open in Colab** 버튼을 클릭한다. 이 노트북은 학생용 읽기와 따라하기 자료다.

이 노트북은 읽기와 따라하기용 강의 노트북이다. 수집한 데이터를 바로 저장하지 않고 스키마, 누락, 중복, 범위 오류를 점검한 뒤 CSV, JSON, SQLite로 남기는 파이프라인을 안전한 합성 fixture로 연습한다.

## 학습 목표

1. 수집 행의 필수 컬럼과 값 범위를 검증한다.
2. 누락, 중복, 형식 오류를 오류 코드로 분류한다.
3. 정제된 데이터와 품질 리포트를 분리 저장한다.
4. CSV, JSON, SQLite 저장 방식의 차이를 설명한다.
5. 저장 전에 검증 로그를 남기는 운영 습관을 만든다.

---

## 1. 수업 맥락과 안전 기준

학원 운영자가 여러 공지/상품/자료실 페이지에서 가져온 데이터를 그대로 쓰면 중복 공지, 가격 형식 오류, 비공개 상태 노출이 섞일 수 있다. 이 레슨은 “수집했다”보다 “믿고 저장할 수 있다”를 기준으로 자동화를 마무리하는 훈련이다.

자동화는 빠르게 반복하는 도구이기 때문에 실패했을 때 더 위험해질 수 있다. 그래서 이번 레슨에서는 모든 입력을 수업용 파일로 고정하고, 결과를 저장하기 전에 검증하거나 로그를 남기는 과정을 코드에 포함한다. 이 습관은 실제 사이트를 대상으로 할 때 요청량을 줄이고, 오류를 빨리 발견하게 만든다.

## 2. 환경 셀

In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/08/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_price(value):
    return clean_int(value)

def parse_stock(value):
    try:
        return int(str(value).strip())
    except ValueError:
        return None

def validate_feed_row(row, rules):
    errors = []
    for field in rules['required_fields']:
        if not str(row.get(field, '')).strip():
            errors.append(f'missing:{field}')
    price = parse_price(row.get('price_text', ''))
    if price <= 0 or price > rules['max_price']:
        errors.append('invalid:price')
    stock = parse_stock(row.get('stock', ''))
    if stock is None or stock < 0:
        errors.append('invalid:stock')
    if row.get('category') not in rules['valid_categories']:
        errors.append('invalid:category')
    if row.get('status') not in rules['valid_status']:
        errors.append('invalid:status')
    return errors


---

## 3. 핵심 개념

이 셀은 원본 피드를 먼저 읽고 검증 규칙과 연결하는 출발점이다. 학생은 행 수와 필수 컬럼을 확인하며 저장 전에 무엇을 검사해야 하는지 정리한다.

In [ ]:
rows = load_csv('raw_product_feed.csv')
rules = load_json('category_rules.json')
print(len(rows), rules['required_fields'])


이 단계 이후에는 원본 행과 정제 행이 분리된다. 학생에게 어떤 행이 제외되었는지 말로 설명하게 한다.

---

## 4. 자료 구조 확인

가격과 재고를 숫자로 바꾸는 과정은 데이터 정제의 기본이다. 출력값을 맞히는 것보다 변환 함수가 실패 값을 어떻게 다루는지 관찰하게 한다.

In [ ]:
sample = rows[0]
print(sample)
print(parse_price(sample['price_text']), parse_stock(sample['stock']))


변환 함수는 실패 값을 조용히 숨기지 않아야 한다. 이상한 입력이 들어왔을 때 어떤 결과가 나오는지 확인한다.

---

## 5. 품질 기준 적용

검증 결과를 행마다 붙이면 오류를 숨기지 않고 추적할 수 있다. 학생은 errors와 is_valid가 서로 어떤 관계인지 설명해야 한다.

In [ ]:
checked = []
for row in rows:
    errors = validate_feed_row(row, rules)
    checked.append({**row, 'errors': '|'.join(errors), 'is_valid': not errors})
print(checked[0])


검증 결과 컬럼은 피드백의 근거가 된다. 교사는 errors 값을 보고 학생의 기준 적용 여부를 빠르게 판단할 수 있다.

---

## 6. 저장과 보고

중복 제거는 저장 직전에 반드시 필요한 단계다. record_id를 기준으로 보되 실제 운영에서는 URL이나 제목까지 묶을 수 있음을 함께 다룬다.

In [ ]:
seen = set()
unique_rows = []
duplicates = []
for row in checked:
    key = row['record_id']
    if key in seen:
        duplicates.append(key)
    else:
        seen.add(key)
        unique_rows.append(row)
print('unique:', len(unique_rows), 'duplicates:', duplicates[:3])


중복 목록은 버리는 데이터가 아니라 품질 리포트의 일부다. 운영자는 중복이 왜 생겼는지 알아야 한다.

---

## 7. 운영 관점 점검

정제 행을 만들 때는 필요한 필드만 남긴다. 원본 문자열과 저장용 숫자를 구분하면 이후 집계와 검색이 쉬워진다.

In [ ]:
clean_rows = []
for row in unique_rows:
    if row['is_valid']:
        clean_rows.append({
            'record_id': row['record_id'],
            'title': row['title'].strip(),
            'category': row['category'],
            'price': parse_price(row['price_text']),
            'stock': parse_stock(row['stock']),
            'status': row['status'],
        })
print(clean_rows[:2])


정제 리스트는 저장 가능한 최소 구조다. 원본 HTML이나 임시 문자열을 그대로 끌고 가지 않는 점이 중요하다.

---

## 8. 마무리 체크

CSV와 JSON 저장은 서로 다른 목적을 가진다. CSV는 상세 목록을 확인하고 JSON은 품질 요약을 공유하는 데 적합하다.

In [ ]:
write_csv('lesson08_clean_feed.csv', clean_rows)
report = {'raw_count': len(rows), 'clean_count': len(clean_rows), 'duplicate_count': len(duplicates)}
Path('lesson08_quality_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report)


저장 파일은 결과 확인용 산출물이다. 파일 존재만 보지 말고 내용 행 수와 헤더를 함께 확인한다.

---

## 9. 핵심 개념

SQLite 예제는 작은 데이터라도 질의 가능한 구조로 저장할 수 있음을 보여준다. 학생에게 category별 집계가 왜 편해지는지 묻는다.

In [ ]:
conn = sqlite3.connect('lesson08_feed.db')
conn.execute('drop table if exists feed')
conn.execute('create table feed(record_id text, title text, category text, price integer, stock integer, status text)')
conn.executemany('insert into feed values(:record_id, :title, :category, :price, :stock, :status)', clean_rows)
print(conn.execute('select category, count(*) from feed group by category').fetchall())
conn.close()


데이터베이스 조회는 저장 후 활용 가능성을 보여준다. 단순 파일 저장보다 질문을 던질 수 있는 구조가 된다.

---

## 데이터 출처와 안전 규칙

raw_product_feed.csv는 의도적으로 누락, 중복, 형식 오류가 섞인 합성 상품 피드다. category_rules.json은 허용 카테고리, 상태, 최대 가격 기준을 담는다. validation_panel.html은 같은 데이터를 웹 표 형태로 확인하는 연습용 페이지다. schema_notes.txt는 사람이 읽는 스키마 설명이다.

- 모든 파일은 수업용 합성 데이터다.
- 실제 사이트에 반복 요청하지 않는다.
- 저장 파일은 레슨 폴더 또는 코랩 현재 작업 폴더에만 만든다.
- 외부 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 포함 여부를 먼저 확인한다.

---

## 수업 운영 메모

이 절은 수업 중 교사가 질문으로 풀어낼 수 있는 운영형 설명이다. 학생이 셀을 실행한 뒤 결과만 맞히지 않고 자동화 절차를 말로 설명하도록 돕는다.

### 1. 검증 우선 원칙

수집 데이터는 저장되기 전까지 임시 결과로 봐야 한다. 필수 컬럼, 값 범위, 허용 category, status를 확인한 뒤에야 운영자가 믿을 수 있는 데이터가 된다.

### 2. 오류 코드 설계

오류를 `invalid` 하나로 뭉치면 나중에 어떤 기준이 문제였는지 알 수 없다. `missing:title`, `invalid:price`처럼 구체적인 코드로 남기면 학생과 교사 모두 피드백이 빠르다.

### 3. 중복 기준

중복은 행 전체가 같은지보다 업무상 같은 항목인지가 중요하다. 이 fixture에서는 record_id를 기준으로 하지만 실제 운영에서는 URL, 제목, 날짜를 함께 묶어 key를 만들 수도 있다.

### 4. 정제 행과 오류 행 분리

유효한 행만 저장하는 것과 오류 행을 버리는 것은 다르다. 오류 행도 별도 리포트에 남겨야 다음 수집 기준을 조정할 수 있다는 점을 강조한다.

### 5. CSV와 JSON의 역할

CSV는 표로 확인하기 좋고 JSON은 요약과 설정을 담기 좋다. 학생에게 두 형식의 장단점을 비교하게 하면 저장 파일을 무작정 하나로 고르지 않게 된다.

### 6. SQLite 도입 이유

작은 수업 데이터라도 SQL로 조회해 보면 필터링과 집계가 왜 필요한지 이해하기 쉽다. 데이터베이스는 큰 시스템이 아니라 구조화된 질문을 던지는 도구로 소개한다.

### 7. 운영 리포트

최종 결과에는 raw_count, clean_count, duplicate_count가 함께 있어야 한다. 숫자 차이를 설명할 수 있어야 수집기가 실제 운영에서 신뢰를 얻는다.

---

## 저장 전 검증 흐름 정리

이번 레슨의 핵심은 저장 버튼을 누르기 전에 데이터가 운영 기준을 통과했는지 확인하는 것이다. 수집 자동화는 화면에서 값을 가져오는 순간보다 저장 직전이 더 중요하다. 저장된 파일은 이후 리포트, 메시지, 대시보드에 재사용되므로 잘못된 행 하나가 여러 화면으로 퍼질 수 있다.

수업에서는 다음 순서를 반복해서 확인한다.

1. 원본 행을 그대로 보관한다.
2. 필수 컬럼과 값 범위를 검사한다.
3. 오류 코드를 행마다 남긴다.
4. 중복 기준을 별도로 적용한다.
5. 정제 행과 오류 리포트를 분리 저장한다.
6. 저장 후 행 수와 요약 값을 다시 확인한다.

학생이 흔히 하는 실수는 오류가 있는 행을 조용히 건너뛰는 것이다. 수업 중에는 “왜 제외되었는지”를 반드시 말하게 한다. 예를 들어 제목이 비었으면 missing:title, 가격이 숫자로 바뀌지 않으면 invalid:price, 같은 record_id가 다시 나오면 duplicate:record_id로 남긴다. 이렇게 남겨야 나중에 원본 페이지 구조가 바뀌었는지, 수집 규칙이 너무 엄격한지 판단할 수 있다.

## CSV, JSON, SQLite 선택 기준

CSV는 사람이 표로 열어 확인하기 쉽고, 학부모 피드백이나 운영 점검처럼 행 단위 검토가 필요한 상황에 적합하다. JSON은 요약 값, 설정값, 오류 통계처럼 구조가 있는 데이터를 저장하기 좋다. SQLite는 저장한 뒤 다시 질문해야 할 때 쓴다. 예를 들어 카테고리별 개수, 재고가 남은 항목, 특정 가격 이상 항목을 반복해서 조회할 때는 SQL이 더 안정적이다.

이번 수업에서는 세 가지 저장 방식을 모두 경험하지만, 학생에게 한 가지 정답을 강요하지 않는다. 대신 어떤 산출물이 누구에게 전달되는지에 따라 형식을 고르게 한다. 운영자가 엑셀로 열어야 하면 CSV, 자동화 다음 단계가 읽어야 하면 JSON, 여러 조건으로 다시 조회해야 하면 SQLite가 자연스럽다.

## 수업 중 확인 질문

- 원본 행 수와 정제 행 수가 다른 이유를 설명할 수 있는가?
- 오류가 있는 행을 삭제하지 않고 코드로 남겼는가?
- 중복 기준을 record_id로 잡은 이유를 설명할 수 있는가?
- 저장 파일을 만든 뒤 실제로 존재 여부와 행 수를 확인했는가?
- 같은 노트북을 다시 실행해도 같은 결과가 나오는가?

이 질문에 답할 수 있으면 학생은 단순 크롤링을 넘어 운영 가능한 자동화의 마무리 과정을 이해한 것이다.


## 검증 코드 읽는 순서

학생이 코드를 읽을 때는 함수 정의부터 외우려 하지 말고 입력 데이터의 흐름을 따라가게 한다. 첫째, raw_product_feed.csv에서 어떤 컬럼이 들어오는지 확인한다. 둘째, category_rules.json이 어떤 기준을 제공하는지 확인한다. 셋째, validate_feed_row가 원본 행과 규칙을 함께 받아 어떤 오류 코드를 만드는지 본다. 넷째, checked와 clean_rows가 서로 다른 목적을 가진 리스트라는 점을 구분한다.

checked는 검증 흔적을 남기기 위한 목록이다. 오류가 있어도 행을 보존하고 errors에 이유를 기록한다. clean_rows는 저장과 조회에 쓸 목록이다. 여기에는 중복이 제거되고, 가격과 재고가 숫자로 바뀐 행만 들어간다. 두 리스트의 목적을 구분하지 못하면 학생은 오류 행을 삭제하거나, 반대로 저장하면 안 되는 행까지 CSV에 넣게 된다.

## 오류 코드를 설계하는 방법

오류 코드는 짧지만 구체적이어야 한다. missing:title은 제목이 비었다는 뜻이고, invalid:category는 허용 목록에 없는 category가 들어왔다는 뜻이다. duplicate:record_id는 데이터 형식 문제가 아니라 저장 기준 문제다. 수업에서는 이 세 종류를 분리해서 설명한다.

- 누락 오류: 필수 값이 비어 있어서 운영자가 읽을 수 없다.
- 형식 오류: 값은 있지만 숫자, 상태, 범위 규칙을 통과하지 못한다.
- 중복 오류: 값 자체는 정상이어도 같은 항목을 두 번 저장할 위험이 있다.

이 구분을 해두면 피드백도 구체적으로 줄 수 있다. “틀렸다”가 아니라 “가격 문자열이 숫자로 변환되지 않았다”, “중복 항목을 저장 직전에 걸러야 한다”처럼 수정 방향이 분명해진다.

## 저장 산출물 검토 방식

레슨 끝에서는 학생이 만든 CSV, JSON, SQLite 파일을 모두 열어볼 필요는 없다. 수업 시간에는 대표 확인만 한다. CSV는 첫 행과 행 수, JSON은 raw_count와 clean_count, SQLite는 select count(*) 결과를 확인한다. 세 값이 서로 연결되면 학생이 저장 과정을 이해했다고 볼 수 있다.

최종 미션을 제출할 때는 파일명만 받지 말고 요약 문장도 함께 받는다. 요약 문장에는 원본 행 수, 정제 행 수, 제외 이유, 다음 실행 시 주의점이 들어가야 한다. 이 네 가지가 들어가면 실제 운영자가 자동화 결과를 다시 확인할 수 있다.


## 실제 운영으로 연결하기

실제 운영 화면에서는 수집 데이터가 한 번 저장되면 여러 기능에서 재사용된다. 예를 들어 공지 목록에서 잘못된 제목을 저장하면 메시지 발송 화면에도 같은 제목이 보이고, 가격이나 상태값이 잘못 저장되면 대시보드 집계도 틀어진다. 그래서 저장 전 검증은 부가 기능이 아니라 자동화의 마지막 안전장치다.

학생에게는 “코드가 돌아갔다”와 “운영에 넣어도 된다”를 분리해서 생각하게 한다. 코드가 돌아가도 필수 컬럼이 비어 있거나, 중복 record_id가 섞였거나, 숫자로 비교해야 할 값이 문자열로 남아 있으면 아직 운영 가능한 결과가 아니다. 이번 레슨의 모든 출력은 이 차이를 확인하기 위한 장치다.

수업 중에는 저장 파일을 만든 뒤 반드시 한 번 더 읽어보게 한다. CSV는 DictReader로 다시 읽어 행 수를 확인하고, JSON은 json.loads로 다시 읽어 key가 남아 있는지 확인한다. SQLite는 select count(*)처럼 가장 단순한 조회부터 실행한다. 저장 후 재검증까지 해보면 학생은 자동화 결과를 더 신중하게 다루게 된다.


---

# 레슨 08 — 실습 문제 정답지

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/08/%EB%A0%88%EC%8A%A8%2008%20%E2%80%94%20%EC%88%98%EC%A7%91%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EA%B2%80%EC%A6%9D%EA%B3%BC%20%EC%A0%80%EC%9E%A5.ipynb)

> 선생님용 강의 노트북이다. 코랩에서 확인하려면 우측 상단 또는 아래의 **Open in Colab** 버튼을 클릭한다.

> 🔒 교사·관리자 전용. 학생에게 배포 금지.

수집 데이터 검증과 저장 실습 문제의 모범 답안이다. 출력값만 보지 말고 검증 기준, 저장 구조, 로그를 함께 확인한다.

## 0. 환경 셀

In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/08/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_price(value):
    return clean_int(value)

def parse_stock(value):
    try:
        return int(str(value).strip())
    except ValueError:
        return None

def validate_feed_row(row, rules):
    errors = []
    for field in rules['required_fields']:
        if not str(row.get(field, '')).strip():
            errors.append(f'missing:{field}')
    price = parse_price(row.get('price_text', ''))
    if price <= 0 or price > rules['max_price']:
        errors.append('invalid:price')
    stock = parse_stock(row.get('stock', ''))
    if stock is None or stock < 0:
        errors.append('invalid:stock')
    if row.get('category') not in rules['valid_categories']:
        errors.append('invalid:category')
    if row.get('status') not in rules['valid_status']:
        errors.append('invalid:status')
    return errors


---

## 문제 1 정답 — 원본 피드 행 수 확인

In [ ]:
rows = load_csv('raw_product_feed.csv')
print(len(rows))


### 왜 이 코드가 정답인지

원본 CSV를 먼저 읽어야 이후 모든 검증의 기준 행 수를 잡을 수 있다. 행 수는 저장 전후 비교의 기준값이므로 별도 변수 rows에 남긴다. 파일명을 추측하지 않고 fixture 이름을 정확히 사용하는 것이 핵심이다.

---

## 문제 2 정답 — 검증 규칙 JSON 읽기

In [ ]:
rules = load_json('category_rules.json')
print(rules['required_fields'])


### 왜 이 코드가 정답인지

검증 규칙은 코드 안에 흩뿌리지 않고 JSON에서 읽는다. required_fields를 출력하면 어떤 컬럼이 반드시 필요한지 학생이 먼저 확인할 수 있고, 이후 validate 함수의 기준과 연결된다.

---

## 문제 3 정답 — 가격 문자열을 숫자로 변환

In [ ]:
price = parse_price(rows[0]['price_text'])
print(price)


### 왜 이 코드가 정답인지

가격 문자열은 쉼표와 원 표시가 섞여 있으므로 숫자만 남기는 변환이 필요하다. price_text를 직접 int로 바꾸면 실패하므로 parse_price를 통해 저장 가능한 정수로 정규화한다.

---

## 문제 4 정답 — 재고 값을 정수로 변환

In [ ]:
stock = parse_stock(rows[0]['stock'])
print(stock)


### 왜 이 코드가 정답인지

재고는 문자열로 읽히지만 저장과 비교에는 정수가 필요하다. parse_stock은 정상 숫자를 int로 바꾸고 실패 입력은 None으로 남겨 검증 단계에서 오류로 분류할 수 있게 한다.

---

## 문제 5 정답 — 첫 행 오류 코드 확인

In [ ]:
errors = validate_feed_row(rows[0], rules)
print(errors)


### 왜 이 코드가 정답인지

한 행을 규칙과 함께 검증해야 누락, 범위, 허용값 오류를 모두 잡을 수 있다. rules를 전달하지 않으면 어떤 category와 status가 허용되는지 알 수 없다.

---

## 문제 6 정답 — 전체 행 검증 결과 만들기

In [ ]:
checked = []
for row in rows:
    errors = validate_feed_row(row, rules)
    checked.append({**row, 'errors': '|'.join(errors), 'is_valid': not errors})
print(len(checked))


### 왜 이 코드가 정답인지

전체 행에 같은 검증 함수를 적용하고 errors와 is_valid를 함께 붙이면 원본과 판정 결과를 한 행에서 추적할 수 있다. 오류를 즉시 삭제하지 않는 점이 운영 자동화에서 중요하다.

---

## 문제 7 정답 — 중복 record_id 찾기

In [ ]:
seen = set()
duplicates = []
for row in checked:
    key = row['record_id']
    if key in seen:
        duplicates.append(key)
    seen.add(key)
print(duplicates[:5])


### 왜 이 코드가 정답인지

중복은 저장 전에 별도로 세야 한다. record_id를 seen set에 모아두면 두 번째 등장한 키만 duplicates에 남길 수 있고, 중복 리포트의 근거가 생긴다.

---

## 문제 8 정답 — 유효 행만 정제 리스트로 만들기

In [ ]:
clean_rows = []
seen = set()
for row in checked:
    if row['record_id'] in seen:
        continue
    seen.add(row['record_id'])
    if row['is_valid']:
        clean_rows.append({'record_id': row['record_id'], 'title': row['title'].strip(), 'category': row['category'], 'price': parse_price(row['price_text']), 'stock': parse_stock(row['stock']), 'status': row['status']})
print(len(clean_rows))


### 왜 이 코드가 정답인지

정제 리스트는 중복을 건너뛰고 검증을 통과한 행만 저장한다. 이때 가격과 재고를 숫자로 바꾸고 필요한 필드만 남기므로 이후 CSV, JSON, SQLite 저장에 바로 사용할 수 있다.

---

## 문제 9 정답 — 카테고리별 개수 요약

In [ ]:
summary = {}
for row in clean_rows:
    summary[row['category']] = summary.get(row['category'], 0) + 1
print(summary)


### 왜 이 코드가 정답인지

카테고리별 개수는 정제 데이터가 실제로 어떤 구성을 갖는지 보여주는 첫 번째 운영 요약이다. 같은 category를 key로 묶고 count를 누적하면 저장 결과를 빠르게 검토할 수 있다.

---

## 문제 10 정답 — 정제 CSV 저장

In [ ]:
write_csv('lesson08_clean_feed.csv', clean_rows)
print(Path('lesson08_clean_feed.csv').exists())


### 왜 이 코드가 정답인지

CSV 저장은 운영자가 표로 확인하기 위한 산출물이다. 저장 후 파일 존재 여부를 바로 출력하면 코드가 실행만 된 것이 아니라 실제 산출물을 만들었는지 확인할 수 있다.

---

## 문제 11 정답 — 품질 리포트 JSON 저장

In [ ]:
report = {'raw_count': len(rows), 'clean_count': len(clean_rows), 'duplicate_count': len(duplicates)}
Path('lesson08_quality_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report)


### 왜 이 코드가 정답인지

품질 리포트에는 원본, 정제, 중복 수가 함께 있어야 한다. 이 값들이 맞아야 왜 일부 행이 제외되었는지 설명할 수 있고, 다음 수집 기준을 조정할 근거가 생긴다.

---

## 문제 12 정답 — SQLite 테이블 생성

In [ ]:
conn = sqlite3.connect('lesson08_feed.db')
conn.execute('drop table if exists feed')
conn.execute('create table feed(record_id text, title text, category text, price integer, stock integer, status text)')
conn.executemany('insert into feed values(:record_id, :title, :category, :price, :stock, :status)', clean_rows)
print(conn.execute('select count(*) from feed').fetchone()[0])
conn.close()


### 왜 이 코드가 정답인지

SQLite는 정제 행을 다시 질의할 수 있는 구조로 바꾼다. 테이블을 만들고 clean_rows를 삽입한 뒤 count를 조회하면 저장된 행 수가 기대와 맞는지 확인할 수 있다.

---

## 문제 13 정답 — 가격 높은 재고 상품 조회

In [ ]:
conn = sqlite3.connect('lesson08_feed.db')
rows_sql = conn.execute('select title, price from feed where stock > 0 and price >= ? order by price desc', (50000,)).fetchall()
print(rows_sql[:5])
conn.close()


### 왜 이 코드가 정답인지

가격과 재고 조건을 SQL 파라미터로 조회하면 저장 후 활용 흐름을 보여줄 수 있다. 문자열 조합 대신 파라미터 바인딩을 사용하면 조건값을 바꿔도 쿼리 구조가 안정적이다.

---

## 문제 14 정답 — HTML 표에서 행 개수 읽기

In [ ]:
soup = BeautifulSoup(load_text('validation_panel.html'), 'html.parser')
items = soup.select('tbody tr')
print(len(items))


### 왜 이 코드가 정답인지

HTML 표 fixture는 웹 화면에서 같은 데이터를 확인하는 상황을 연습한다. tbody tr를 선택하면 헤더가 아니라 실제 데이터 행만 세므로 CSV 행 수와 비교할 수 있다.

---

## 문제 15 정답 — 검증 파이프라인 함수 만들기

In [ ]:
def build_clean_feed():
    rows = load_csv('raw_product_feed.csv')
    rules = load_json('category_rules.json')
    checked = []
    seen = set()
    clean = []
    for row in rows:
        errors = validate_feed_row(row, rules)
        if row['record_id'] in seen:
            errors.append('duplicate:record_id')
        seen.add(row['record_id'])
        checked.append({**row, 'errors': '|'.join(errors), 'is_valid': not errors})
        if not errors:
            clean.append({'record_id': row['record_id'], 'title': row['title'].strip(), 'category': row['category'], 'price': parse_price(row['price_text']), 'stock': parse_stock(row['stock']), 'status': row['status']})
    return checked, clean
checked_rows, clean_rows = build_clean_feed()
print(len(checked_rows), len(clean_rows))


### 왜 이 코드가 정답인지

파이프라인 함수는 읽기, 검증, 중복 판정, 정제를 한 번에 재실행 가능하게 묶는다. checked와 clean을 같이 반환해야 오류 리포트와 저장용 데이터를 동시에 관리할 수 있다.

## 교사용 검토 메모

이번 정답지는 값 하나를 맞히는 용도가 아니라 데이터 검증 파이프라인을 확인하기 위한 기준이다. 학생 답안이 모범 코드와 조금 달라도 다음 조건을 만족하면 인정할 수 있다.

- 필수 컬럼 검증이 누락되지 않았다.
- 가격과 재고가 저장 전에 숫자로 정규화되었다.
- 오류 행을 조용히 삭제하지 않고 errors 또는 별도 리포트에 남겼다.
- 중복 record_id를 정제 저장 대상에서 제외했다.
- CSV 또는 JSON 산출물을 만들고 존재 여부를 확인했다.
- 최종 함수가 다시 실행되어도 같은 행 수를 만든다.

반대로 출력값이 우연히 맞아도 검증 기준이 코드에 드러나지 않거나, 오류 행을 설명 없이 버렸다면 다시 보완하게 한다. 운영 자동화에서는 성공 결과보다 실패를 추적하는 구조가 더 중요하다.


## 문제별 채점 포인트

### 문제 1 채점 포인트

학생이 원본 피드 파일명을 정확히 입력했는지 확인한다. 행 수만 맞아도 변수에 rows를 남기지 않으면 이후 문제에서 다시 읽어야 하므로 감점할 수 있다. 원본 행 수는 정제 행 수와 비교하는 기준이다.

### 문제 2 채점 포인트

category_rules.json에서 required_fields를 직접 읽는지 확인한다. 필수 컬럼 목록을 코드에 직접 하드코딩하면 현재 문제는 맞아도 규칙 파일이 바뀔 때 자동화가 따라가지 못한다.

### 문제 3 채점 포인트

가격 문자열을 숫자로 변환하는 이유를 설명할 수 있어야 한다. 쉼표나 원 표시가 남아 있으면 SQLite 조회와 가격 비교가 불가능하므로 parse_price 사용 여부를 본다.

### 문제 4 채점 포인트

재고는 음수와 잘못된 문자열을 잡아야 한다. parse_stock이 실패 값을 None으로 돌려주고, validate 단계에서 invalid:stock으로 분류되는 흐름을 설명하면 충분하다.

### 문제 5 채점 포인트

첫 행 검증은 함수 호출 형태를 보는 문제다. row만 넣거나 rules만 넣으면 검증 기준이 완성되지 않는다. errors가 빈 리스트일 수 있다는 것도 정상 결과로 인정한다.

### 문제 6 채점 포인트

checked 리스트에 원본 필드와 판정 필드가 함께 들어가는지 확인한다. errors를 문자열로 합치는 이유는 CSV나 표에서 사람이 읽기 쉽게 만들기 위한 것이다. is_valid는 not errors로 계산해야 한다.

### 문제 7 채점 포인트

중복 record_id를 발견한 뒤 즉시 삭제하지 않고 duplicates에 기록하는지 본다. 중복 목록이 있어야 품질 리포트에서 왜 정제 행 수가 줄었는지 설명할 수 있다.

### 문제 8 채점 포인트

정제 리스트에는 유효 행만 들어가야 하며 중복은 한 번만 저장되어야 한다. title은 strip으로 공백을 정리하고 price, stock은 숫자로 저장하는지 확인한다.

### 문제 9 채점 포인트

카테고리 요약은 저장 결과의 균형을 확인하는 간단한 집계다. summary.get(row['category'], 0) + 1 흐름을 이해하면 다른 상태값 요약으로도 확장할 수 있다.

### 문제 10 채점 포인트

CSV 저장 후 파일 존재를 확인하는 습관을 본다. write_csv만 호출하고 끝내면 실제 저장 성공 여부가 보이지 않는다. 산출물 파일명은 문제에서 지정한 이름을 유지해야 한다.

### 문제 11 채점 포인트

JSON 리포트는 운영 요약이다. raw_count, clean_count, duplicate_count가 모두 있어야 원본과 결과 차이를 설명할 수 있다. ensure_ascii=False를 사용하면 한글 리포트가 읽기 좋다.

### 문제 12 채점 포인트

SQLite 테이블 생성은 저장 후 조회 가능성을 보여준다. drop table로 재실행성을 확보하고, executemany에 clean_rows를 넣어야 같은 노트북을 다시 실행해도 깨지지 않는다.

### 문제 13 채점 포인트

SQL 조건 조회는 저장 데이터가 실제 질문에 답할 수 있는지 확인한다. stock > 0과 price >= ? 조건을 함께 사용하면 판매 가능한 고가 항목을 찾는 운영 예시가 된다.

### 문제 14 채점 포인트

HTML 표에서는 tbody tr를 선택해야 데이터 행만 센다. table tr 전체를 선택하면 헤더가 있는 HTML에서는 행 수가 달라질 수 있음을 짚어준다.

### 문제 15 채점 포인트

최종 함수는 읽기, 검증, 중복 처리, 정제를 재사용 가능한 단위로 묶어야 한다. checked와 clean을 모두 반환해야 오류 추적과 저장이 동시에 가능하다. 함수 안에서 파일명을 다시 읽기 때문에 다른 노트북에서도 독립적으로 실행된다.

## 오답 유형과 피드백 문장

- CSV를 바로 저장한 경우: “저장 전에 검증 결과를 남겨야 다음 실행에서 제외 이유를 설명할 수 있다.”
- 오류 행을 삭제한 경우: “오류 행도 리포트에는 남겨야 원본 품질을 개선할 수 있다.”
- 중복을 세지 않은 경우: “정제 행 수가 줄어든 이유를 duplicate_count로 설명해야 한다.”
- 문자열 가격을 그대로 저장한 경우: “가격 비교와 SQL 조회를 하려면 숫자로 정규화해야 한다.”
- 파일 존재 확인이 없는 경우: “자동화는 산출물이 실제로 만들어졌는지 마지막에 확인해야 한다.”

## 수업 후 점검 기준

이번 레슨을 마친 학생은 “수집 데이터는 바로 저장하지 않는다”는 원칙을 말할 수 있어야 한다. 또한 원본 행, 검증 행, 정제 행, 저장 파일이 서로 다른 역할을 가진다는 점을 구분해야 한다. 이 구분이 되면 이후 9강의 알림/리포트 자동화에서도 잘못된 데이터를 보내는 위험을 줄일 수 있다.


## 단계별 해설 보충

### 원본과 규칙을 분리해서 읽는 이유

원본 데이터와 검증 규칙을 같은 파일에 섞어두면 자동화가 커질수록 관리가 어려워진다. 이번 레슨에서는 raw_product_feed.csv가 바뀌어도 category_rules.json의 기준은 그대로 유지될 수 있고, 반대로 운영 기준이 바뀌면 JSON만 바꿔 다시 실행할 수 있다. 학생이 이 구조를 이해하면 이후 수업에서 요청 간격, 허용 도메인, 저장 경로 같은 설정도 별도 파일로 분리할 수 있다.

### 정규화 함수의 역할

parse_price와 parse_stock은 단순한 편의 함수가 아니다. 저장 전에 데이터 타입을 확정하는 단계다. 가격이 문자열로 남아 있으면 “5만원 이상” 같은 조건을 안정적으로 처리할 수 없고, 재고가 문자열로 남아 있으면 음수 검증도 불안정해진다. 함수가 작아 보여도 운영 자동화에서는 입력을 저장 가능한 형태로 바꾸는 경계 역할을 한다.

### validate_feed_row 설계 의도

검증 함수는 한 행을 받아 오류 목록을 돌려준다. 참/거짓 하나만 돌려주지 않는 이유는 학생과 운영자가 실패 원인을 알아야 하기 때문이다. 예를 들어 제목도 비었고 category도 잘못된 행은 두 오류를 모두 남겨야 한다. 하나만 남기면 첫 번째 오류를 고친 뒤 두 번째 오류를 다시 발견하게 되어 피드백 시간이 길어진다.

### checked와 clean_rows를 나누는 이유

checked에는 원본과 오류 정보가 함께 들어간다. 이 목록은 피드백과 품질 리포트에 쓰인다. clean_rows에는 운영 저장에 필요한 필드만 들어간다. 이 목록은 CSV, JSON, SQLite에 쓰인다. 두 목록을 하나로 합치면 오류 행을 저장하거나, 반대로 오류 리포트를 잃어버리는 문제가 생긴다. 그래서 수업에서는 두 변수 이름을 계속 비교해서 읽게 한다.

### 중복 처리 위치

중복은 validate_feed_row 안에 넣을 수도 있지만, 이 레슨에서는 별도 반복문과 최종 함수에서 처리한다. 이유는 중복 기준이 필드 값의 형식 오류와 성격이 다르기 때문이다. price나 status는 한 행만 봐도 검증할 수 있지만, duplicate는 앞에서 같은 record_id가 나왔는지 기억해야 판단할 수 있다. 이 차이를 이해하면 학생이 상태를 기억하는 set의 필요성을 자연스럽게 받아들인다.

### 저장 형식별 피드백

CSV를 저장한 학생에게는 헤더와 행 수를 먼저 보게 한다. JSON을 저장한 학생에게는 key 이름과 값의 의미를 설명하게 한다. SQLite를 저장한 학생에게는 직접 쿼리를 하나 더 만들어보게 한다. 같은 데이터라도 저장 형식이 달라지면 검토 방법도 달라진다는 점이 이번 레슨의 중요한 포인트다.

### 최종 함수의 재실행성

build_clean_feed는 셀을 다시 실행해도 같은 결과를 만들어야 한다. 함수 내부에서 rows, rules, checked, seen, clean을 새로 만들기 때문에 이전 실행 상태에 덜 의존한다. 학생이 전역 변수에만 기대면 노트북 실행 순서가 바뀌었을 때 결과가 달라질 수 있다. 함수로 묶는 연습은 코랩 수업에서도 실무적인 안정성을 높인다.

## 예상 출력 확인

원본 피드는 오류와 중복을 포함한 40행이다. 유효하면서 중복이 아닌 정제 행은 그보다 적다. 학생 답안에서 정확한 숫자가 다르다면 먼저 validate_feed_row 기준을 확인한다. 가격문의, 빈 제목, private category, 음수 stock, hidden status, 중복 P006이 제대로 제외되었는지 보면 대부분의 오류를 찾을 수 있다.

HTML 표 행 수는 CSV 전체 행 수와 다를 수 있다. validation_panel.html은 일부 행만 웹 패널에 표시하는 fixture이므로, 학생이 CSV 행 수와 HTML 행 수를 무조건 같다고 가정하면 안 된다. 이 차이는 실제 운영 화면과 원본 수집 데이터가 항상 1:1로 대응하지 않는다는 설명으로 연결한다.

## 교사용 빠른 판정표

| 항목 | 통과 기준 | 다시 볼 신호 |
| --- | --- | --- |
| 원본 읽기 | rows가 CSV 전체 행을 담는다 | 파일명을 잘못 입력하거나 행 수가 0이다 |
| 규칙 읽기 | required_fields와 valid 목록을 JSON에서 읽는다 | 기준을 코드에 직접 적었다 |
| 정규화 | 가격과 재고가 숫자로 바뀐다 | 원 표시, 쉼표가 남아 있다 |
| 검증 | errors에 구체적 코드가 남는다 | True/False만 남기고 이유가 없다 |
| 중복 | duplicate 목록이나 코드가 있다 | 중복 P006이 그대로 저장된다 |
| 저장 | CSV/JSON/SQLite 중 산출물이 생긴다 | 저장 호출 후 확인 출력이 없다 |
| 요약 | raw, clean, duplicate 수를 설명한다 | 행 수 차이를 말하지 못한다 |

## 수업 마무리 피드백 예시

좋은 답안은 코드가 길지 않아도 기준이 분명하다. “필수 컬럼을 확인하고, 가격과 재고를 숫자로 바꾼 뒤, 오류 행과 중복 행을 저장 대상에서 제외했다”라고 설명할 수 있으면 핵심을 이해한 것이다. 보완이 필요한 답안은 대체로 저장 파일은 만들었지만 왜 그 행들이 저장되었는지 설명하지 못한다. 이때는 정답 코드를 보여주기보다 errors와 duplicate_count를 직접 출력하게 하는 편이 효과적이다.


## 재실행 검증 기준

교사용 노트북은 한 번 실행된 상태에서만 맞으면 충분하지 않다. 커널을 재시작한 뒤 위에서 아래로 다시 실행해도 같은 CSV, JSON, SQLite 파일이 만들어져야 한다. 특히 SQLite 셀은 drop table을 먼저 실행하므로 같은 셀을 반복 실행해도 테이블 중복 오류가 나지 않는다. CSV와 JSON은 같은 파일명을 덮어쓰는 구조라서 수업 중 여러 번 실행해도 산출물 이름이 늘어나지 않는다.

학생 답안에서 이전 셀의 우연한 상태에 기대는 코드가 보이면 재실행 검증으로 바로 드러난다. 예를 들어 rows나 rules를 만들지 않고 중간 문제부터 실행하면 실패하는 것은 자연스럽지만, 전체 실행에서도 실패한다면 변수 생성 순서가 잘못된 것이다. 최종 피드백은 “한 셀만 맞히기”보다 “전체 노트북을 다시 실행해도 같은 결과가 나오는지”를 기준으로 준다.


---

# 레슨 08 — 최종 미션 모범 답안

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/08/%EB%A0%88%EC%8A%A8%2008%20%E2%80%94%20%EC%88%98%EC%A7%91%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EA%B2%80%EC%A6%9D%EA%B3%BC%20%EC%A0%80%EC%9E%A5.ipynb)

> 선생님용 최종 미션 모범 답안이다. 코랩에서 확인하려면 우측 상단 또는 아래의 **Open in Colab** 버튼을 클릭한다.

> 교사 확인용 모범 답안이다. 학생에게는 최종 미션 조건만 제공한다.

## 실행 코드

In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/08/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_price(value):
    return clean_int(value)

def parse_stock(value):
    try:
        return int(str(value).strip())
    except ValueError:
        return None

def validate_feed_row(row, rules):
    errors = []
    for field in rules['required_fields']:
        if not str(row.get(field, '')).strip():
            errors.append(f'missing:{field}')
    price = parse_price(row.get('price_text', ''))
    if price <= 0 or price > rules['max_price']:
        errors.append('invalid:price')
    stock = parse_stock(row.get('stock', ''))
    if stock is None or stock < 0:
        errors.append('invalid:stock')
    if row.get('category') not in rules['valid_categories']:
        errors.append('invalid:category')
    if row.get('status') not in rules['valid_status']:
        errors.append('invalid:status')
    return errors

rows = load_csv('raw_product_feed.csv')
print(len(rows))

rules = load_json('category_rules.json')
print(rules['required_fields'])

price = parse_price(rows[0]['price_text'])
print(price)

stock = parse_stock(rows[0]['stock'])
print(stock)

errors = validate_feed_row(rows[0], rules)
print(errors)

checked = []
for row in rows:
    errors = validate_feed_row(row, rules)
    checked.append({**row, 'errors': '|'.join(errors), 'is_valid': not errors})
print(len(checked))

seen = set()
duplicates = []
for row in checked:
    key = row['record_id']
    if key in seen:
        duplicates.append(key)
    seen.add(key)
print(duplicates[:5])

clean_rows = []
seen = set()
for row in checked:
    if row['record_id'] in seen:
        continue
    seen.add(row['record_id'])
    if row['is_valid']:
        clean_rows.append({'record_id': row['record_id'], 'title': row['title'].strip(), 'category': row['category'], 'price': parse_price(row['price_text']), 'stock': parse_stock(row['stock']), 'status': row['status']})
print(len(clean_rows))

summary = {}
for row in clean_rows:
    summary[row['category']] = summary.get(row['category'], 0) + 1
print(summary)

write_csv('lesson08_clean_feed.csv', clean_rows)
print(Path('lesson08_clean_feed.csv').exists())

report = {'raw_count': len(rows), 'clean_count': len(clean_rows), 'duplicate_count': len(duplicates)}
Path('lesson08_quality_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report)

conn = sqlite3.connect('lesson08_feed.db')
conn.execute('drop table if exists feed')
conn.execute('create table feed(record_id text, title text, category text, price integer, stock integer, status text)')
conn.executemany('insert into feed values(:record_id, :title, :category, :price, :stock, :status)', clean_rows)
print(conn.execute('select count(*) from feed').fetchone()[0])
conn.close()

conn = sqlite3.connect('lesson08_feed.db')
rows_sql = conn.execute('select title, price from feed where stock > 0 and price >= ? order by price desc', (50000,)).fetchall()
print(rows_sql[:5])
conn.close()

soup = BeautifulSoup(load_text('validation_panel.html'), 'html.parser')
items = soup.select('tbody tr')
print(len(items))

def build_clean_feed():
    rows = load_csv('raw_product_feed.csv')
    rules = load_json('category_rules.json')
    checked = []
    seen = set()
    clean = []
    for row in rows:
        errors = validate_feed_row(row, rules)
        if row['record_id'] in seen:
            errors.append('duplicate:record_id')
        seen.add(row['record_id'])
        checked.append({**row, 'errors': '|'.join(errors), 'is_valid': not errors})
        if not errors:
            clean.append({'record_id': row['record_id'], 'title': row['title'].strip(), 'category': row['category'], 'price': parse_price(row['price_text']), 'stock': parse_stock(row['stock']), 'status': row['status']})
    return checked, clean
checked_rows, clean_rows = build_clean_feed()
print(len(checked_rows), len(clean_rows))


## 채점 메모

- 입력 파일을 모두 읽었는지 확인한다.
- 검증 기준이 코드에 명시되어 있는지 확인한다.
- 저장 파일과 운영 요약이 함께 있는지 확인한다.


---

# 레슨 08 — 교사 가이드

## 학습 목표 상세

- 수집 행의 필수 컬럼과 값 범위를 검증한다.
- 누락, 중복, 형식 오류를 오류 코드로 분류한다.
- 정제된 데이터와 품질 리포트를 분리 저장한다.
- CSV, JSON, SQLite 저장 방식의 차이를 설명한다.
- 저장 전에 검증 로그를 남기는 운영 습관을 만든다.

## 수업 전 준비

- 코랩 버튼이 학생용과 선생님용으로 각각 열리는지 확인한다.
- `data/` 폴더의 fixture 파일을 먼저 훑고, 학생에게 실제 사이트가 아니라 합성 데이터임을 설명한다.
- 레슨 08의 핵심은 수집한 데이터를 바로 저장하지 않고 스키마, 누락, 중복, 범위 오류를 점검한 뒤 CSV, JSON, SQLite로 남기는 파이프라인이다.

## 2시간 운영안

1. 0~15분: 오늘의 자동화 실패 사례와 안전 기준 설명.
2. 15~45분: 강의 노트북 예제 실행.
3. 45~90분: 15문제 중 1~10번 풀이.
4. 90~110분: 11~15번과 저장 산출물 확인.
5. 110~120분: 최종 미션 안내와 제출 기준 정리.

## 학생이 자주 막히는 지점

- 파일명, 컬럼명, selector를 추측해서 오타가 난다.
- 저장 전에 검증하지 않고 바로 CSV를 만든다.
- 실패 상태를 예외나 로그로 남기지 않는다.

## 피드백 기준

15문제 중 12문제 이상 통과를 기본 완료로 본다. 최종 미션은 산출물 파일과 3문장 요약이 함께 있어야 완료 처리한다. 정답 코드와 다른 방식이어도 입력 구조, 검증 기준, 출력 형태가 맞으면 인정한다.

## 심화 질문

- 이 자동화를 실제 사이트에 적용하면 요청 간격은 어떻게 바꿔야 할까?
- 어떤 오류는 재시도하고 어떤 오류는 바로 멈춰야 할까?
- 저장 파일을 운영자가 다시 읽을 때 가장 필요한 컬럼은 무엇일까?

## 마무리 체크리스트

- 학생이 fixture 출처를 설명할 수 있다.
- 학생이 빈칸을 채운 이유를 말할 수 있다.
- 학생이 저장 파일을 열어 행 수를 확인했다.
- 학생이 다음 수업에서 개선할 점을 한 문장으로 남겼다.

## 운영 판서 흐름

수업 시작 시 칠판에는 “수집 -> 검증 -> 정제 -> 저장 -> 보고” 순서를 먼저 적는다. 학생이 코드를 실행하기 전에 각 단계에서 무엇을 확인하는지 말하게 하면 뒤 문제의 빈칸이 자연스럽게 연결된다.

- 수집: 원본 행 수와 헤더 확인
- 검증: 필수 컬럼, 가격, 재고, category, status 확인
- 정제: 필요한 필드만 남기고 숫자형으로 변환
- 저장: CSV, JSON, SQLite 중 목적에 맞게 선택
- 보고: 원본 수, 정제 수, 중복 수, 오류 이유 요약

## 채점 시 우선순위

1. 검증 함수가 실제로 모든 행에 적용되었는지 확인한다.
2. 중복 record_id를 따로 기록했는지 확인한다.
3. 저장 파일이 만들어졌고 행 수를 확인했는지 확인한다.
4. 최종 미션 요약이 운영자가 읽을 수 있는 문장인지 확인한다.

학생이 속도에 집중하면 오류 행을 빨리 버리는 방향으로 답안을 작성하기 쉽다. 이때는 “버린 행을 다시 설명할 수 있나?”라고 질문한다. 설명할 수 없다면 자동화 결과를 운영에 사용할 수 없다는 점을 짚어준다.

## 확장 활동

시간이 남으면 학생에게 category_rules.json의 max_price를 바꿔 다시 실행하게 한다. 규칙 하나가 바뀌면 clean_count와 오류 코드가 어떻게 바뀌는지 관찰할 수 있다. 또는 validation_panel.html의 행 수와 CSV 행 수를 비교하게 하여 웹 화면과 원본 데이터가 항상 같은 것은 아니라는 점을 설명한다.


## 수업 중 질문 예시

- 원본 행 수와 정제 행 수가 같지 않은 이유는 무엇인가?
- errors 컬럼을 문자열로 남기는 이유는 무엇인가?
- record_id 중복은 값 오류인가, 저장 기준 오류인가?
- CSV와 JSON 중 학부모에게 전달하기 좋은 형식은 무엇인가?
- SQLite에 저장하면 어떤 질문을 더 쉽게 할 수 있는가?

## 제출물 확인 루틴

학생 제출물을 볼 때는 노트북 전체를 처음부터 다시 읽기보다 산출물 기준으로 확인한다. lesson08_clean_feed.csv가 있으면 첫 3행과 행 수를 본다. lesson08_quality_report.json이 있으면 raw_count와 clean_count가 문제 풀이 결과와 맞는지 본다. SQLite 파일이 있으면 count 조회가 되는지만 확인한다.

## 보충 설명이 필요한 학생

검증 함수는 작성했지만 저장 전에 적용하지 않은 학생은 흐름을 놓친 것이다. 이 경우 함수 자체를 고치게 하기보다, checked를 만드는 반복문 위치를 다시 찾게 한다. 반대로 저장은 했지만 리포트가 없는 학생에게는 “운영자가 왜 40행 중 일부만 저장되었는지 알 수 있나?”라고 질문한다.